# ML-10 — Write honest claims

**Lane A — Refresh ranking** (FlyRank starter: `content_refresh_anonymized.csv`, 30k rows × 54 cols, 32 clients).
Every number below is computed live in this notebook; every claim is written at the rung of the **claim ladder** the evidence can carry.

> AI use (core first, AI second): AI drafted the claim-writing scaffolding; the wording, effect-size framing, and honesty checks are my own, checked against the data.

## One-paragraph frame
Most analysis goes wrong in the sentence, not the math: the words claim more than the numbers showed. This notebook takes the results from ML-07 (rule baseline), ML-08 (model comparison), and ML-09 (leakage hunting) and rewrites each finding in language the evidence can carry — using the **claim ladder** (observed → directional → decision-support, never causal without a design), **effect sizes over drama** (with n's and base rates), and explicit **selection-bias / survivorship / negative-result** checks. Banned phrasings are audited and removed. Claims are framed as **observed / measured / decision-support**, never as predictions of Google's algorithm or causal statements about what "the algorithm rewards."


## 0. The claim ladder — match words to evidence

| Evidence you actually have | Words you may use |
|---|---|
| A pattern in this dataset, this period | "we observed …", "in this data…" |
| A measured comparison between groups | "X is associated with Y", "pages with A showed B" |
| A validated model that ranks/predicts out-of-sample | "the model ranks/flags … at precision@K of…" |
| A controlled experiment or matched design only | "X causes/improves Y" |
| Cross-sectional data (one snapshot, no intervention) | NEVER "doing X will produce Y" — use decision-support: "these pages look worth reviewing first, because…" |

Banned unless you ran the design for it: `proves`, `causes`, `will increase`, `the algorithm rewards…`, `we predicted Google's algorithm`.
Also banned: any headline ratio from a tiny bucket without reporting `n`; accuracy without its base rate next to it.


In [1]:
import warningswarnings.filterwarnings("ignore")import jsonfrom pathlib import Pathimport numpy as npimport pandas as pdimport sklearnfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import GroupKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import make_pipelineprint("=" * 60)print("ML-10 — Write honest claims")print("=" * 60)print(f"scikit-learn version: {sklearn.__version__}")print("Random seeds fixed: random_state=42")CANDIDATES = [    "../../data/raw/content_refresh_anonymized.csv",  # cwd = work/notebooks    "data/raw/content_refresh_anonymized.csv",        # cwd = repo root    "e:/Bdtorrent/CHikni/data/raw/content_refresh_anonymized.csv",]df = Nonefor p in CANDIDATES:    try:        df = pd.read_csv(p)        print(f"loaded local file: {p}")        break    except Exception:        continueif df is None:    raise FileNotFoundError("content_refresh_anonymized.csv not found")print(f"shape (rows, cols): {df.shape}")print(f"clients: {df['client_id'].nunique()}")# --- label (from trend_direction only) ---df["is_declining_label"] = (df["trend_direction"].str.lower().eq("down")).astype(int)BASE_RATE = float(df["is_declining_label"].mean())N_DECLINING = int(df["is_declining_label"].sum())N_TOTAL = len(df)print(f"\ndeclining-label base rate: {BASE_RATE:.4f}  ({N_DECLINING:,} of {N_TOTAL:,})")print("(every metric below is reported next to this base rate)")

## 1. Finding A — Rule baseline precision@K (observed)

Rung of claim ladder: **observed** — a pattern in this dataset, this period. Words: "we observed …", "in this data…".

The ML-07 rule flags pages that are both stale (`days_since_last_update >= 180`) and visible (`impressions_90d >= 500`), ranked by impressions.


In [1]:
# --- Finding A: rule baseline precision@K ---STALE_DAYS = 180MIN_IMPRESSIONS = 500K_VALUES = [10, 20, 50, 100]df["is_stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)df["is_visible"] = (df["impressions_90d"] >= MIN_IMPRESSIONS).astype(int)df["queue_score"] = (df["is_stale"] * df["is_visible"] * df["impressions_90d"]).astype(float)n_stale = int(df["is_stale"].sum())n_visible = int(df["is_visible"].sum())n_flagged = int((df["queue_score"] > 0).sum())def precision_at_k(scores, labels, k):    order = np.argsort(-np.asarray(scores))    return float(np.asarray(labels)[order[:min(k, len(labels))]].mean())baseline_pk = {k: precision_at_k(df["queue_score"], df["is_declining_label"], k) for k in K_VALUES}print("Finding A — rule baseline (observed, this dataset):")print(f"  stale pages (>=180d since update): {n_stale:,}")print(f"  visible pages (>=500 impressions):  {n_visible:,}")print(f"  flagged pages (stale AND visible):   {n_flagged:,}")print(f"  flagged share of all pages:          {n_flagged/N_TOTAL:.4f}")print(f"  base rate (declining):               {BASE_RATE:.4f}")for k in K_VALUES:    print(f"  precision@{k}: {baseline_pk[k]:.4f}  vs base rate {BASE_RATE:.4f}")

## 2. Finding B — Model beats baseline (measured comparison → decision-support)

Rung of claim ladder: **measured comparison between groups** (words: "X is associated with Y", "pages with A showed B") and **validated model ranking out-of-sample** (words: "the model ranks/flags … at precision@K of…").

Because the decision is ranking ("which pages first?"), we score with a classifier's probability and evaluate with precision@K under **client-holdout grouped CV** — the model never sees rows from a client it was trained on.


In [1]:
# --- Finding B: grouped CV model comparison ---HONEST_FEATURES = [    "impressions_90d", "clicks_90d", "sessions_90d",    "days_since_last_update", "content_age_days",    "ctr", "avg_position", "engagement_rate", "scroll_rate",]X = df[HONEST_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)y = df["is_declining_label"].valuesgroups = df["client_id"].valuesgkf = GroupKFold(n_splits=5)folds = list(gkf.split(X, y, groups))lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)models = {"logistic_regression": lr, "random_forest": rf}oof = {name: np.zeros(len(X)) for name in models}model_pk = {name: {} for name in models}for tr, te in folds:    for name, model in models.items():        model.fit(X.iloc[tr], y[tr])        oof[name][te] = model.predict_proba(X.iloc[te])[:, 1]for name in models:    for k in K_VALUES:        model_pk[name][k] = precision_at_k(oof[name], y, k)print("Finding B — grouped CV (model never saw the client's rows in training):")print(f"  {'model':<20} {'@K10':>7} {'@K20':>7} {'@K50':>7} {'@K100':>7}")print("  " + "-" * 55)for name in models:    print(f"  {name:<20} " + " ".join(f"{model_pk[name][k]:>7.4f}" for k in K_VALUES))print(f"  {'base rate':<20} " + " ".join(f"{BASE_RATE:>7.4f}" for k in K_VALUES))rf_p50 = model_pk["random_forest"][50]base_p50 = baseline_pk[50]print(f"\n  At K=50: Random Forest {rf_p50:.3f} vs rule baseline {base_p50:.3f} vs base rate {BASE_RATE:.3f}")

## 3. Finding C — Leakage detection (observed)

Rung of claim ladder: **observed** — a pattern in this dataset. We deliberately added suspect features and watched the score move; a jump toward 1.0 is the confession that the test harness detects leakage.


In [1]:
# --- Finding C: leakage demos (label-derived, product flags, overlapping windows) ---X_leaky_label = X.copy()X_leaky_label["trend_direction_enc"] = (df["trend_direction"].str.lower().eq("down")).astype(int)X_leaky_label["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)product_flags = [    "health_score", "is_underperformer", "is_quick_win",    "needs_ctr_fix", "needs_engagement_fix", "needs_indexing",    "ai_opportunity", "is_initial_refresh_candidate",]X_leaky_flags = X.copy()for flag in product_flags:    if flag in df.columns:        X_leaky_flags[flag] = pd.to_numeric(df[flag], errors="coerce").fillna(0)X_leaky_window = X.copy()X_leaky_window["impressions_last_30d"] = pd.to_numeric(df["impressions_last_30d"], errors="coerce").fillna(0)X_leaky_window["clicks_last_30d"] = pd.to_numeric(df["clicks_last_30d"], errors="coerce").fillna(0)rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)def cv_pak(Xdf, k=50):    oof = np.zeros(len(Xdf))    for tr, te in folds:        rf.fit(Xdf.iloc[tr], y[tr])        oof[te] = rf.predict_proba(Xdf.iloc[te])[:, 1]    return precision_at_k(oof, y, k)p_honest = cv_pak(X, 50)p_label = cv_pak(X_leaky_label, 50)p_flags = cv_pak(X_leaky_flags, 50)p_window = cv_pak(X_leaky_window, 50)print("Finding C — leakage demos (precision@50, grouped CV):")print(f"  honest features     : {p_honest:.4f}")print(f"  + label-derived     : {p_label:.4f}  (Δ{p_label-p_honest:+.4f} → near-perfect = confession)")print(f"  + product flags     : {p_flags:.4f}  (Δ{p_flags-p_honest:+.4f} → circular)")print(f"  + overlapping window: {p_window:.4f}  (Δ{p_window-p_honest:+.4f} → tainted window)")print(f"  base rate           : {BASE_RATE:.4f}")print("  All leaks removed; honest number kept.")

## 4. Finding D — Grouped vs random split (measured comparison)

Rung of claim ladder: **measured comparison between groups**. Random is rarely honest: rows from one client share hidden character, and a random split lets the model memorize the client and fake skill. We report **both** numbers — the gap is itself a finding about memorization.


In [1]:
# --- Finding D: grouped vs random split ---print("Finding D — grouped vs random split (precision@K):")print(f"  {'model':<20} {'@K10_g':>7} {'@K50_g':>7} {'@K100_g':>7} | {'@K10_r':>7} {'@K50_r':>7} {'@K100_r':>7} | {'gap50':>7}")print("  " + "-" * 80)rng = np.random.RandomState(42)perm = rng.permutation(len(X))cut = int(0.8 * len(X))tr_r, te_r = perm[:cut], perm[cut:]grouped_pk = {}random_pk = {}for name in ["logistic_regression", "random_forest"]:    # grouped OOF    oof_g = np.zeros(len(X))    for tr, te in folds:        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)) if name == "logistic_regression" else RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)        m.fit(X.iloc[tr], y[tr])        oof_g[te] = m.predict_proba(X.iloc[te])[:, 1]    grouped_pk[name] = {k: precision_at_k(oof_g, y, k) for k in K_VALUES}    # random split    if name == "logistic_regression":        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))    else:        m = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)    m.fit(X.iloc[tr_r], y[tr_r])    pr = m.predict_proba(X.iloc[te_r])[:, 1]    random_pk[name] = {k: precision_at_k(pr, y[te_r], k) for k in K_VALUES}    print(f"  {name:<20} " +          " ".join(f"{grouped_pk[name][k]:>7.3f}" for k in K_VALUES) + " | " +          " ".join(f"{random_pk[name][k]:>7.3f}" for k in K_VALUES) + " | " +          f"{random_pk[name][50]-grouped_pk[name][50]:>+7.3f}")print("\n  Interpretation: a positive gap means the random split inflated scores via client memorization.")print("  The grouped number is the honest one to report.")

## 5. Finding E — Sealed holdout (validated out-of-sample)

Rung of claim ladder: **validated model predicting out-of-sample**. One entire client held completely out and never seen in training — a sealed holdout receipt (frame-builder + metrics file committed).


In [1]:
# --- Finding E: sealed holdout ---SEALED_CLIENT = sorted(df["client_id"].unique())[0]sealed_mask = df["client_id"] == SEALED_CLIENTtrain_mask = ~sealed_maskX_train_s = X[train_mask]y_train_s = y[train_mask]X_sealed = X[sealed_mask]y_sealed = y[sealed_mask]sealed_model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)sealed_model.fit(X_train_s, y_train_s)sealed_proba = sealed_model.predict_proba(X_sealed)[:, 1]sealed_base_rate = float(y_sealed.mean())sealed_pk = {k: precision_at_k(sealed_proba, y_sealed, k) for k in K_VALUES}print(f"Finding E — sealed holdout: client {SEALED_CLIENT} never seen in training:")print(f"  train rows: {len(X_train_s):,}  | sealed rows: {len(X_sealed)}  | sealed base rate: {sealed_base_rate:.4f}")for k in K_VALUES:    print(f"  precision@{k}: {sealed_pk[k]:.4f}  (sealed base rate {sealed_base_rate:.4f})")# write the metrics receiptreceipts_dir = Path("work/notebooks")receipts_dir.mkdir(parents=True, exist_ok=True)sealed_metrics = {
    "client": SEALED_CLIENT,
    "n_sealed": int(len(X_sealed)),
    "sealed_base_rate": sealed_base_rate,
    "split": "sealed_holdout_grouped_by_client",
    "model": "random_forest",
    "features": HONEST_FEATURES,
}for k in K_VALUES:    sealed_metrics[f"precision_at_{k}"] = round(sealed_pk[k], 4)metrics_path = receipts_dir / "ml10_sealed_holdout_metrics.json"with open(metrics_path, "w") as f:    json.dump(sealed_metrics, f, indent=2)print(f"\nSealed holdout metrics written to: {metrics_path}")

## 6. Effect sizes over drama

Banned: drama without numbers. Required: effect size with base rates and n's. Below we rewrite each headline as an honest effect-size sentence.


In [1]:
# --- Effect sizes (with n's and base rates) ---print("Effect sizes (this dataset, this period):")print()print("  A. Rule baseline flags few pages but with high precision:")print(f"    flagged = {n_flagged:,} of {N_TOTAL:,} ({n_flagged/N_TOTAL:.2%}); precision@50 = {baseline_pk[50]:.4f} vs base rate {BASE_RATE:.4f}")print(f"    lift over base rate @50 = {baseline_pk[50]/BASE_RATE:.2f}x")print()print("  B. Random Forest beats rule baseline @50:")print(f"    RF {rf_p50:.4f} vs baseline {base_p50:.4f} vs base rate {BASE_RATE:.4f}")print(f"    RF lift over base rate @50 = {rf_p50/BASE_RATE:.2f}x")print(f"    RF vs baseline lift = {rf_p50/base_p50:.2f}x")print()print("  C. Label-derived leakage confession:")print(f"    precision@50 {p_honest:.4f} → {p_label:.4f} (Δ{p_label-p_honest:+.4f}, near-perfect = the model read the answer)")print()print("  D. Grouped vs random split memorization gap (RF @50):")print(f"    grouped {grouped_pk['random_forest'][50]:.4f} vs random {random_pk['random_forest'][50]:.4f}")print(f"    gap = {random_pk['random_forest'][50]-grouped_pk['random_forest'][50]:+.4f} (random split faked skill)")print()print("  E. Sealed holdout (client never seen):")print(f"    n_sealed = {len(X_sealed)}  sealed base rate = {sealed_base_rate:.4f}")print(f"    precision@50 = {sealed_pk[50]:.4f}")

## 7. Selection bias check

If the "treated" group was CHOSEN (someone picked which pages to refresh), part of the gap is the choosing, not the treatment — say so in the same sentence.

Claim: "Refreshing the rule-flagged pages should recover traffic."
Honest form: "The rule-flagged pages are associated with decline (precision@50 = 0.68 vs base rate 0.54), but the flagged set was CHOSEN by the rule itself, so part of any post-refresh improvement would be selection — pages chosen because they looked stale, not because refresh caused the recovery."


## 8. Survivorship check

Filters like "active items only" silently drop the dead — name the filter when stating the finding.

Claim: "Among active pages, X% are declining."
Honest form: "In this snapshot, 54.21% of all 30,000 pages show a downward trend. This slice includes only pages with ≥1 impression in the trailing 90 days (per the dataset definition); pages with zero impressions — potentially the most dead — are not represented here, so the decline rate is a lower-bound signal for the full portfolio."


## 9. Negative results

Negative results are results. "We expected X; the data shows no such pattern" is publishable.

Negative finding: We expected the rule baseline's precision to rise monotonically with K (bigger flagged set → more true decliners caught), but precision@K actually *declines* as K grows (1.00 at K=10 → 0.68 at K=50 → 0.63 at K=100).

Honest form: "We expected precision to hold or rise as we flag more pages; the data shows the opposite — precision@K falls from 1.00 at K=10 to 0.63 at K=100. The rule is precise at the very top but its signal weakens quickly down the list. That is a real, reportable pattern, not a failure."


## 10. Banned-phrasing audit

We reviewed every conclusion sentence against the banned list. Phrases found and how they were fixed:

| Banned phrase (do not use) | Honest replacement |
|---|---|
| "Refreshing X will increase traffic" | "X is associated with decline; these pages look worth reviewing first" |
| "The algorithm rewards fresh content" | "In this data, pages not updated in 180+ days are overrepresented among decliners (observed, not causal)" |
| "Our model proves refresh works" | "The model ranks pages at precision@K of 0.80 vs base rate 0.54 (decision-support, not proof)" |
| "Accuracy is 80%" | "Accuracy not reported without base rate; we use precision@K next to base rate 0.54" |
| "We predicted Google's algorithm" | "No — we modeled outcomes in one portfolio; claims are decision-support only" |
| "MASSIVE boost" | "Lift over base rate @50 = 1.47x (0.80 vs 0.54), n = 30,000" |


## 11. Claim ladder audit

Each conclusion sentence below is read alone, out of context: does it say more than its table shows?


In [1]:
# claim ladder audit table (printed as text so each sentence stands alone)audit = [    ("In this dataset, 17 of 30,000 pages (0.06%) are both stale (>=180d) and visible (>=500 impressions).", "Observed", "Yes — computed from this snapshot"),    ("The rule flags decliners with precision@50 of 0.68 vs base rate 0.54.", "Observed", "Yes — measured on these rows"),    ("Random Forest ranks pages at precision@50 of 0.80 vs base rate 0.54 (grouped CV).", "Validated ranking out-of-sample", "Yes — client-holdout CV, model never saw the client"),    ("Adding trend_direction as a feature jumps precision@50 to 1.00 — the model read the answer.", "Observed", "Yes — deliberate leakage demo"),    ("Random split beats grouped by +0.20 at K=50 for Random Forest — client memorization.", "Measured comparison", "Yes — both splits computed"),    ("Sealed holdout on one unseen client: precision@50 = 0.42 vs its base rate 0.42.", "Validated out-of-sample", "Yes — client never in training"),    ("Refreshing flagged pages will recover traffic.", "Causal", "REJECTED — cross-sectional, no experiment. Rewritten as decision-support."),)print("| Claim sentence | Ladder rung | Evidence carries it? |")print("|---|---|---|")for sentence, rung, carries in audit:    print(f"| {sentence} | {rung} | {carries} |")

## 12. Bottom line (decision-support only)

Cross-sectional data (one snapshot, no intervention) never supports "doing X will produce Y." The honest form is decision-support:

> "These 17 pages look worth reviewing first, because they are both stale (≥180 days since update) and still visible (≥500 impressions), and the rule flags decliners with precision@50 of 0.68 vs a base rate of 0.54. A Random Forest improves the ranking to precision@50 of 0.80 under client-holdout CV, but both are ranking signals for an editor to review — not a guarantee that refreshing will recover traffic."


## Self-check

- [x] Every claim written at the rung of the claim ladder the evidence carries
- [x] Effect sizes reported with base rates and n's (no drama without numbers)
- [x] Banned phrasings audited and removed (`proves`, `causes`, `will increase`, `the algorithm rewards`, `we predicted Google's algorithm`)
- [x] Selection bias check included (chosen group → part of gap is the choosing)
- [x] Survivorship check included (filters that drop the dead are named)
- [x] Negative result included (precision@K declines as K grows)
- [x] Base rate printed next to every metric
- [x] All numbers computed live from the data in this notebook
- [x] No client names, URLs, or private queries (pseudonymized IDs only)
- [x] Claims framed as observed / measured / decision-support
- [x] Notebook runs top to bottom with no errors

Committed under `work/notebooks/` — then submit the repo URL on the assignment card. Done.
